### Cell 1: Initialization and Configuration

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from kg_pipeline import KGPipeline
from df_helpers import ResolveCoreferences, ExtractConcepts, df2Graph, graph2Df, contextual_proximity

# --- config ---
INPUT_FILE = "../data_input/dataset/hotpot/hotpot_dev_distractor_v1.json"
OUTPUT_DIR = "../data_output/dataset/hotpot/ds10_2" # 建议修改输出目录，避免覆盖之前的结果

START_INDEX = 0  # Starting Position
END_INDEX = 10    # Ending Position (None means to the end)
# Whether to force regeneration (True: ignore cache and re-run; False: prioritize loading cache)
REGENERATE = True  

# Initialize Pipeline instance
pipeline = KGPipeline(output_dir=OUTPUT_DIR)

d:\Anaconda_envs\envs\andelie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1 - Chunking

In [2]:
# Load the original JSON data and split it into Chunks
if REGENERATE:
    df_chunks = pipeline.load_and_split_data(
        json_path=INPUT_FILE, 
        start_index=START_INDEX, # 新增参数
        end_index=END_INDEX
    )
else:
    #  Try to load the cache
    chunk_cache = Path(OUTPUT_DIR) / "chunk.csv"
    if chunk_cache.exists():
        df_chunks = pd.read_csv(chunk_cache, sep="|")
    else:
        # If the cache does not exist, force re-generation
        df_chunks = pipeline.load_and_split_data(json_path=INPUT_FILE, start_index=START_INDEX, end_index=END_INDEX)

print(f"Loaded {len(df_chunks)} chunks.")
print(df_chunks.head(2))

🚀 [Pipeline] 开始数据分块...
Loaded 103 chunks.
   context_id  chunk_id             title  \
0           0         0    Ed Wood (film)   
1           0         1  Scott Derrickson   

                                                text  
0  Ed Wood (film) information: Ed Wood is a 1994 ...  
1  Scott Derrickson information: Scott Derrickson...  


## Step 2 - Coreference Resolution

In [3]:
# Resolve coreferences for the Chunks
# REGENERATE = True
resolved_path = Path(OUTPUT_DIR) / "resolved_chunks.csv"

if REGENERATE or not resolved_path.exists():
    df_chunks = ResolveCoreferences(df_chunks,max_workers=10)
    df_chunks.to_csv(resolved_path, sep="|", index=False)
else:
    print("Loading resolved chunks from cache...")
    df_chunks = pd.read_csv(resolved_path, sep="|")

print(df_chunks[['text', 'resolved_text']].head(2))

--- Step: Starting parallel coreference resolution on 103 chunks ---
🚀 Submitting 103 tasks to thread pool...
⏳ Starting parallel processing...


100%|██████████| 103/103 [01:15<00:00,  1.36chunk/s]


🧹 Cleaning up thread pool (dropping stuck threads)...
✅ Processing finished. Successfully obtained results: 103/103
✅ Coreference resolution complete.
                                                text  \
0  Ed Wood (film) information: Ed Wood is a 1994 ...   
1  Scott Derrickson information: Scott Derrickson...   

                                       resolved_text  
0  Ed Wood (film) information: Ed Wood is a 1994 ...  
1  Scott Derrickson (born July 16, 1966) is an Am...  


## Step 3 -chunks Embeddings 

In [4]:
# REGENERATE = True
final_emb_filename = "chunks_with_embeddings.parquet"
temp_title_filename = "temp_title_embeddings.parquet" # 临时文件
final_path = Path(OUTPUT_DIR) / final_emb_filename
temp_path = Path(OUTPUT_DIR) / temp_title_filename

# 1. The calculation will be recalculated only when REGENERATE is set to True; otherwise, it will be loaded directly.
if REGENERATE or not final_path.exists():
    print("🚀 [Notebook] 开始双重向量计算 (Content + Title)...")

    # --- Step 1: Calculate the content vector ---
    # This will generate the 'embedding' column
    df_content = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='text', 
        id_col='chunk_id', 
        file_name=final_emb_filename 
    )

    # --- Step 2: Calculate the title vector ---
    # In order not to overwrite the original file, we first save it to a temporary file.
    # Note: pipeline defaults to saving the column name as 'embedding'
    df_title = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='title', 
        id_col='chunk_id', 
        file_name=temp_title_filename 
    )

    # --- Step 3: Merge and Rename in Notebook ---
    print("🔄 [Notebook] 正在合并 Title 向量...")
    
    # Extract the chunk_id and embedding from df_title, and rename the embedding to title_embedding.
    # Note: Here we only take the required two columns to prevent other columns from being duplicated
    df_title_subset = df_title[['chunk_id', 'embedding']].copy()
    df_title_subset.rename(columns={'embedding': 'title_embedding'}, inplace=True)
    
    df_final = pd.merge(df_content, df_title_subset, on='chunk_id', how='left')
    
    # --- Step 4: Manually save the merged results ---
    # Must convert numpy array back to list to save as Parquet (this is what Pipeline does internally, we do it manually outside)
    for col in ['embedding', 'title_embedding']:
        if col in df_final.columns:
            if not df_final[col].empty and isinstance(df_final[col].iloc[0], np.ndarray):
                df_final[col] = df_final[col].apply(lambda x: x.tolist())

    df_final.to_parquet(final_path, index=False)
    
    if temp_path.exists():
        os.remove(temp_path)
        
    print(f"✅ [Notebook] Double vector merging completed! Saved to {final_path}")
    df_chunks = df_final # Update the df_chunks currently stored in the memory

else:
    print(f"🔍 [Notebook] Loading existing cache: {final_path}")
    df_chunks = pd.read_parquet(final_path)

# --- Validate data ---
print("-" * 30)
first_row = df_chunks.iloc[0]
print(f"Data Loaded. Columns: {df_chunks.columns.tolist()}")
if 'embedding' in df_chunks.columns:
    print(f"Content Vector Dim: {len(first_row['embedding'])}")
if 'title_embedding' in df_chunks.columns:
    print(f"Title Vector Dim:   {len(first_row['title_embedding'])}")

🚀 [Notebook] 开始双重向量计算 (Content + Title)...
⏳ [Pipeline] 开始计算 103 条数据的嵌入向量...


Computing Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.17s/it]


✅ [Pipeline] 保存至 ..\data_output\dataset\hotpot\ds10_2\chunks_with_embeddings.parquet
⏳ [Pipeline] 开始计算 103 条数据的嵌入向量...


Computing Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.05s/it]

✅ [Pipeline] 保存至 ..\data_output\dataset\hotpot\ds10_2\temp_title_embeddings.parquet
🔄 [Notebook] 正在合并 Title 向量...
✅ [Notebook] Double vector merging completed! Saved to ..\data_output\dataset\hotpot\ds10_2\chunks_with_embeddings.parquet
------------------------------
Data Loaded. Columns: ['context_id', 'chunk_id', 'title', 'text', 'resolved_text', 'embedding', 'title_embedding']
Content Vector Dim: 1024
Title Vector Dim:   1024


## Step 4 -Concept Extraction

In [ ]:
# REGENERATE = False
# Extract entities from the text in parallel
concepts_path = Path(OUTPUT_DIR) / "extracted_concepts.csv"

if REGENERATE or not concepts_path.exists():
    df_concepts = ExtractConcepts(dataframe=df_chunks, max_workers=10)
    df_concepts.to_csv(concepts_path, sep="|", index=False)
else:
    df_concepts = pd.read_csv(concepts_path, sep="|")

print(f"Extracted {len(df_concepts)} concepts.")

--- Step: Starting parallel concept extraction on 103 chunks ---
🚀 Submitting 103 tasks to thread pool...
⏳ Starting parallel processing...


 73%|███████▎  | 75/103 [01:29<00:44,  1.58s/chunk]

⚠️ Overall parsing failed, attempting to extract objects one by one...
✅ Robust extraction successfully recovered 11 records.


100%|██████████| 103/103 [02:26<00:00,  1.42s/chunk]


🧹 Cleaning up thread pool (dropping stuck threads)...
✅ Processing finished. Successfully obtained results: 103/103
✅ Concept extraction complete, extracted 1364 entities.
Extracted 1364 concepts.


## Step 5 - Entity Standardization

In [6]:
# This step includes the most complex logic:
# 1. Aggregating entity synonyms (Rich Text)
# 2. Computing entity embeddings
# 3. Aligning using HAC + Genealogy Penalty
# 4. Generating standard entity names
# This step includes the most complex logic: aggregating synonyms -> computing embeddings -> HAC + Genealogy Penalty -> generating standard names
# REGENERATE = False

std_path = Path(OUTPUT_DIR) / "dp_extracted_concepts.csv"
if REGENERATE or not std_path.exists():
    # # Execute the standardization process (the files will be saved internally)
    df_concepts_std = pipeline.standardize_entities(df_concepts)
else:
    print(f"🔍 Loading cached standardized entities: {std_path}")
    df_concepts_std = pd.read_csv(std_path, sep="|")

print("Standardization sample:")
print(df_concepts_std[['Entity', 'Standard_Entity', 'cluster_id']].head())

🚀 [Pipeline] 开始实体标准化流程...
前5条展开：    context_id  chunk_id                Entity category  \
0           0         0           Bela Lugosi   Person   
1           0         0     Patricia Arquette   Person   
2           0         0  Sarah Jessica Parker   Person   
3           0         0           Johnny Depp   Person   
4           0         0            Tim Burton   Person   

                                         description                synonyms  
0  The actor who played a role in Ed Wood's films...           [Bela Lugosi]  
1  One of the supporting cast members in the film...     [Patricia Arquette]  
2  One of the supporting cast members in the film...  [Sarah Jessica Parker]  
3  The actor who stars as cult filmmaker Ed Wood ...           [Johnny Depp]  
4     The director and producer of the film Ed Wood.            [Tim Burton]  
分组后前5条：    context_id Entity category Aggregated_Synonyms  \
0           0   1931     Time                1931   
1           0   1948     Time 

Computing Embeddings: 100%|██████████| 20/20 [00:36<00:00,  1.85s/it]


✅ [Pipeline] 保存至 ..\data_output\dataset\hotpot\ds10_2\entity_embeddings.parquet
⏳ [Pipeline] 执行智能分组聚类 (HAC + Genealogy Penalty)...


Clustering: 100%|██████████| 110/110 [00:01<00:00, 81.56it/s]



⏳ Starting post‑processing of 'Person' type full names / abbreviations...
✅ Post‑processing complete. Updated 16 mapping relationships.
🚀 [Pipeline] 应用实体映射 (Key: Context + Entity + Category)...
✅ [Pipeline] 标准化完成。
Standardization sample:
                 Entity       Standard_Entity  cluster_id
0     Patricia Arquette     Patricia Arquette          43
1  Sarah Jessica Parker  Sarah Jessica Parker          41
2           Johnny Depp           Johnny Depp          49
3           Bela Lugosi           Bela Lugosi          60
4        Cult filmmaker        Cult filmmaker          76


## Step 6 - QA 数据提取

In [7]:
# Extracting questions and answers pairs from the original data
qa_path = Path(OUTPUT_DIR) / "qa.csv"


REGENERATE =True
if REGENERATE or not qa_path.exists():
    df_qa = pipeline.extract_qa_pairs(INPUT_FILE, max_contexts=END_INDEX-START_INDEX)
else:
    print(f"🔍 Loading cached QA data: {qa_path}")
    df_qa = pd.read_csv(qa_path, sep="|")
REGENERATE =False

print(df_qa.head(2))

                                            question             answer  \
0  Were Scott Derrickson and Ed Wood of the same ...                yes   
1  What government position was held by the woman...  Chief of Protocol   

   context_id  
0           0  
1           1  


### Step 7 - 基础图谱构建 (Relation Extraction)

In [ ]:
# 1. Prepare the entity mapping dictionary {chunk: {original_name: standard_name}}
entity_map = pipeline.generate_entity_map_for_graph(df_concepts_std)

# REGENERATE = True

# 2. Perform relation extraction
graph_path = Path(OUTPUT_DIR) / "graph.csv"

if REGENERATE or not graph_path.exists():
    # Use LLM to extract relationships and pass them into entity_map to achieve guided extraction
    relations_list = df2Graph(df_chunks, entity_map,max_workers=10)
    df_graph = graph2Df(relations_list)
    df_graph.to_csv(graph_path, sep="|", index=False)
else:
    df_graph = pd.read_csv(graph_path, sep="|")

print(f"Extracted {len(df_graph)} relations.")
print(df_graph.head())

d:\Desktop\ID-SGTR-main\ID-SGTR-main\knowledge_graph\hotpot\kg_pipeline.py:388: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return valid.groupby(['context_id', 'chunk_id']).apply(


--- Step: Starting parallel relation extraction ---
🚀 Submitting 103 tasks to thread pool...
⏳ Starting parallel processing...


 72%|███████▏  | 74/103 [01:03<00:26,  1.08chunk/s]

### Step 8 - 计算上下文共现关系 (Contextual Proximity)

In [ ]:
# Calculate additional graph edges based on Chunk co-occurrence
# REGENERATE = False

df_proximity = contextual_proximity(df_graph)
df_proximity.to_csv(Path(OUTPUT_DIR) / "contextual_proximity.csv", sep="|", index=False)

print(f"Generated {len(df_proximity)} proximity edges.")
print(df_proximity.head())

Generated 254096 proximity edges.
   context_id                node_1                   node_2 chunk_id  count  \
7         320      " I Like My Job"             Duke Tumatoe     3255     46   
38        638      "1 Night 2 Days"             Lee Seung-gi     6427     22   
50          6  "100 Latinos Madrid"       Amaruk Kayshapanta       71     18   
66        314                  "19"  "Critics' Choice" award     3194      6   
67        314                  "19"         "Hometown Glory"     3194     12   

                    edge  
7   contextual_proximity  
38  contextual_proximity  
50  contextual_proximity  
66  contextual_proximity  
67  contextual_proximity  


## Step 9 - Merge Entities & Compute Dual Embeddings

In [ ]:
# --- New Step: Merge and Enrich ---
# Aggregate entity information and calculate vectors (vec_entity and vec_desc)
merged_path = Path(OUTPUT_DIR) / "concepts_merged_with_vectors.parquet"

if REGENERATE and merged_path.exists():
    print(f"🗑️ [REGENERATE] 删除旧聚合缓存: {merged_path}")
    os.remove(merged_path)

# Call the Pipeline (internal logic: calculate and save if no file exists, otherwise load)
df_merged_vectors = pipeline.merge_and_embed_concepts(df_concepts=df_concepts_std)

print("Merged Data Sample:")
print(df_merged_vectors[['Standard_Entity', 'description', 'vec_entity']].head(2))

🗑️ [REGENERATE] 删除旧聚合缓存: ..\data_output\dataset\hotpot\ds1000\concepts_merged_with_vectors.parquet
🚀 [Pipeline] 开始实体聚合...
🔄 [Helper] 正在聚合实体数据...
✅ [Helper] 聚合完成。原始行数: 125497 -> 聚合后行数: 104057
🛠️ [Pipeline] 构建增强实体文本...
🚀 [Pipeline] 计算 Enriched Entity Vectors...


Vec: Entity: 100%|██████████| 1626/1626 [26:40<00:00,  1.02it/s]


🚀 [Pipeline] 计算 Description Vectors...


Vec: Desc: 100%|██████████| 1626/1626 [27:01<00:00,  1.00it/s]


💾 [Pipeline] 保存聚合向量表到: ..\data_output\dataset\hotpot\ds1000\concepts_merged_with_vectors.parquet
Merged Data Sample:
        Standard_Entity                                     description  \
0            Tim Burton  The director and producer of the film Ed Wood.   
1  Sarah Jessica Parker    A member of the supporting cast of the film.   

                                          vec_entity  
0  [-0.011303096, 0.007036083, 0.026462099, 0.027...  
1  [0.004416721, -0.0030213601, -0.021587733, 0.0...  
